# Healthcare Phrase Validation Matrix for KSL Translation Project

## Notebook 07: Validation Preparation for Kenyan Sign Language Healthcare Phrases

This notebook comes after:

| Notebook | Stage |
|---|---|
| 01 Random Forest | Traditional recognition baseline |
| 02 HMM | Traditional temporal baseline |
| 03 LSTM | Deep temporal baseline |
| 04 ST-GCN | Graph-based recognition |
| 05 Transformer | Attention-based recognition |
| 06 Gloss-to-healthcare text | Language translation baseline |

Now the project moves into **validation preparation**.

---

# Why this notebook is needed

Your final project is not just an AI model.

It is a healthcare communication framework for:

```text
Kenyan Sign Language users ↔ healthcare workers
```

So the phrases must be validated for:

- medical usefulness
- KSL clarity
- patient safety
- cultural/local relevance
- technical feasibility
- usability in real healthcare settings

This notebook creates the structured validation files needed for:

1. KSL interpreter review
2. Deaf/KSL user review
3. Healthcare worker review
4. ICT/AI expert review
5. Thesis and Paper 3 validation evidence

---

# What this notebook produces

The notebook generates:

| Output file | Purpose |
|---|---|
| `healthcare_phrase_validation_matrix.csv` | Main validation matrix for all healthcare phrases |
| `expert_review_questionnaire.csv` | Questionnaire items for reviewers |
| `validation_scoring_template.csv` | Template for entering reviewer scores |
| `validation_summary_template.csv` | Summary template for mean score, agreement, and status |
| `validation_metadata.json` | Metadata explaining what was generated |

---

# Important note

The phrases in this notebook are **starter validation phrases**.

They are not final validated KSL translations yet.

They must be reviewed by:

- KSL interpreter
- deaf/KSL user
- healthcare worker
- academic/ICT expert

## 1. Import libraries

We use:

- `pandas` for tables
- `numpy` for numeric scoring
- `json` for metadata
- `Path` for file paths

In [1]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

print("Validation matrix notebook started successfully")

Validation matrix notebook started successfully


## 2. Configure project folders

Outputs are saved under:

```text
/workspace/dataset/ksl_project_data/validation/
```

If the notebook is not running in Docker, it falls back to:

```text
ksl_project_data/validation/
```

In [2]:
PROJECT_DIR = Path("/workspace/dataset/ksl_project_data").resolve()

if not Path("/workspace/dataset").exists():
    PROJECT_DIR = Path("ksl_project_data").resolve()

VALIDATION_DIR = PROJECT_DIR / "validation"
REPORT_DIR = PROJECT_DIR / "reports"

for folder in [PROJECT_DIR, VALIDATION_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Validation directory:", VALIDATION_DIR)
print("Report directory:", REPORT_DIR)

Project directory: /workspace/dataset/ksl_project_data
Validation directory: /workspace/dataset/ksl_project_data/validation
Report directory: /workspace/dataset/ksl_project_data/reports


## 3. Load phrase dataset from Notebook 06 if available

Notebook 06 created:

```text
language_translation/healthcare_gloss_phrase_dataset.csv
```

This notebook tries to load that file first.

If the file is not found, it creates a fallback phrase dataset.

In [3]:
phrase_dataset_path = PROJECT_DIR / "language_translation" / "healthcare_gloss_phrase_dataset.csv"

if phrase_dataset_path.exists():
    phrases_df = pd.read_csv(phrase_dataset_path)
    print("Loaded phrase dataset from:", phrase_dataset_path)
else:
    print("Phrase dataset not found. Creating fallback starter dataset.")

    fallback_phrases = [
        {"id": 1, "domain": "triage", "gloss": "pain where", "healthcare_text": "Where is the pain?", "priority": "critical"},
        {"id": 2, "domain": "emergency", "gloss": "you chest pain have", "healthcare_text": "Do you have chest pain?", "priority": "critical"},
        {"id": 3, "domain": "medicine", "gloss": "medicine allergy you", "healthcare_text": "Are you allergic to any medicine?", "priority": "critical"},
        {"id": 4, "domain": "pharmacy", "gloss": "medicine take day two", "healthcare_text": "Take this medicine twice a day.", "priority": "important"},
        {"id": 5, "domain": "follow_up", "gloss": "come back week one", "healthcare_text": "Come back after one week.", "priority": "normal"},
        {"id": 6, "domain": "reception", "gloss": "doctor wait please", "healthcare_text": "Please wait for the doctor.", "priority": "normal"},
        {"id": 7, "domain": "emergency", "gloss": "breathing difficult you", "healthcare_text": "Do you have difficulty breathing?", "priority": "critical"},
        {"id": 8, "domain": "assessment", "gloss": "pregnant you", "healthcare_text": "Are you pregnant?", "priority": "important"},
        {"id": 9, "domain": "medicine", "gloss": "medicine today take you", "healthcare_text": "Have you taken medicine today?", "priority": "important"},
        {"id": 10, "domain": "diagnosis", "gloss": "hurt where show me", "healthcare_text": "Show me where it hurts.", "priority": "critical"},
    ]

    phrases_df = pd.DataFrame(fallback_phrases)

phrases_df

Loaded phrase dataset from: /workspace/dataset/ksl_project_data/language_translation/healthcare_gloss_phrase_dataset.csv


,id,domain,gloss,healthcare_text,priority
0,1,triage,pain where,Where is the pain?,critical
1,2,emergency,you chest pain have,Do you have chest pain?,critical
2,3,medicine,medicine allergy you,Are you allergic to any medicine?,critical
3,4,pharmacy,medicine take day two,Take this medicine twice a day.,important
4,5,follow_up,come back week one,Come back after one week.,normal
5,6,reception,doctor wait please,Please wait for the doctor.,normal
6,7,emergency,breathing difficult you,Do you have difficulty breathing?,critical
7,8,assessment,pregnant you,Are you pregnant?,important
8,9,medicine,medicine today take you,Have you taken medicine today?,important
9,10,diagnosis,hurt where show me,Show me where it hurts.,critical


## 4. Create healthcare phrase validation matrix

This is the main validation table.

Each phrase will be reviewed for:

| Field | Meaning |
|---|---|
| `healthcare_text` | Natural healthcare sentence |
| `gloss` | Draft gloss/sign-style representation |
| `domain` | Healthcare context |
| `priority` | Clinical risk level |
| `ksl_interpreter_review` | Whether KSL meaning is correct |
| `deaf_user_review` | Whether KSL user understands it |
| `healthcare_worker_review` | Whether phrase is medically useful/safe |
| `ict_ai_review` | Whether system can support it technically |
| `validation_status` | Pending, approved, needs revision, rejected |

The matrix starts with all review fields marked as `pending`.

In [4]:
validation_rows = []

for _, row in phrases_df.iterrows():
    healthcare_text = row["healthcare_text"]
    priority = row["priority"]

    if priority == "critical":
        risk_level = "high"
        validation_required = "mandatory"
    elif priority == "important":
        risk_level = "medium"
        validation_required = "recommended"
    else:
        risk_level = "low"
        validation_required = "recommended"

    validation_rows.append(
        {
            "phrase_id": row["id"],
            "domain": row["domain"],
            "priority": priority,
            "risk_level": risk_level,
            "validation_required": validation_required,
            "gloss": row["gloss"],
            "healthcare_text": healthcare_text,
            "clinical_use_case": "",
            "ksl_interpreter_review": "pending",
            "ksl_interpreter_comments": "",
            "deaf_user_review": "pending",
            "deaf_user_comments": "",
            "healthcare_worker_review": "pending",
            "healthcare_worker_comments": "",
            "ict_ai_review": "pending",
            "ict_ai_comments": "",
            "revision_needed": "",
            "final_validated_gloss": "",
            "validation_status": "pending",
        }
    )

validation_matrix_df = pd.DataFrame(validation_rows)
validation_matrix_df

,phrase_id,domain,priority,risk_level,validation_required,gloss,healthcare_text,clinical_use_case,ksl_interpreter_review,ksl_interpreter_comments,deaf_user_review,deaf_user_comments,healthcare_worker_review,healthcare_worker_comments,ict_ai_review,ict_ai_comments,revision_needed,final_validated_gloss,validation_status
0,1,triage,critical,high,mandatory,pain where,Where is the pain?,,pending,,pending,,pending,,pending,,,,pending
1,2,emergency,critical,high,mandatory,you chest pain have,Do you have chest pain?,,pending,,pending,,pending,,pending,,,,pending
2,3,medicine,critical,high,mandatory,medicine allergy you,Are you allergic to any medicine?,,pending,,pending,,pending,,pending,,,,pending
3,4,pharmacy,important,medium,recommended,medicine take day two,Take this medicine twice a day.,,pending,,pending,,pending,,pending,,,,pending
4,5,follow_up,normal,low,recommended,come back week one,Come back after one week.,,pending,,pending,,pending,,pending,,,,pending
5,6,reception,normal,low,recommended,doctor wait please,Please wait for the doctor.,,pending,,pending,,pending,,pending,,,,pending
6,7,emergency,critical,high,mandatory,breathing difficult you,Do you have difficulty breathing?,,pending,,pending,,pending,,pending,,,,pending
7,8,assessment,important,medium,recommended,pregnant you,Are you pregnant?,,pending,,pending,,pending,,pending,,,,pending
8,9,medicine,important,medium,recommended,medicine today take you,Have you taken medicine today?,,pending,,pending,,pending,,pending,,,,pending
9,10,diagnosis,critical,high,mandatory,hurt where show me,Show me where it hurts.,,pending,,pending,,pending,,pending,,,,pending


## 5. Add clinical use cases

This cell fills a short clinical use-case description based on the phrase domain.

These descriptions help reviewers understand why each phrase matters.

In [5]:
domain_use_cases = {
    "triage": "Initial patient assessment and symptom location",
    "emergency": "Emergency screening and urgent response",
    "medicine": "Medication history and patient safety",
    "pharmacy": "Medication counselling and dosage instructions",
    "follow_up": "Follow-up instruction and care continuity",
    "reception": "Patient registration and facility guidance",
    "assessment": "Clinical assessment and patient history",
    "diagnosis": "Symptom clarification and diagnosis support",
}

validation_matrix_df["clinical_use_case"] = validation_matrix_df["domain"].map(domain_use_cases).fillna(
    "General healthcare communication"
)

validation_matrix_df[["phrase_id", "domain", "healthcare_text", "clinical_use_case", "risk_level"]]

,phrase_id,domain,healthcare_text,clinical_use_case,risk_level
0,1,triage,Where is the pain?,Initial patient assessment and symptom location,high
1,2,emergency,Do you have chest pain?,Emergency screening and urgent response,high
2,3,medicine,Are you allergic to any medicine?,Medication history and patient safety,high
3,4,pharmacy,Take this medicine twice a day.,Medication counselling and dosage instructions,medium
4,5,follow_up,Come back after one week.,Follow-up instruction and care continuity,low
5,6,reception,Please wait for the doctor.,Patient registration and facility guidance,low
6,7,emergency,Do you have difficulty breathing?,Emergency screening and urgent response,high
7,8,assessment,Are you pregnant?,Clinical assessment and patient history,medium
8,9,medicine,Have you taken medicine today?,Medication history and patient safety,medium
9,10,diagnosis,Show me where it hurts.,Symptom clarification and diagnosis support,high


## 6. Save healthcare phrase validation matrix

This is the main file you can give to reviewers.

In [6]:
validation_matrix_path = VALIDATION_DIR / "healthcare_phrase_validation_matrix.csv"
validation_matrix_df.to_csv(validation_matrix_path, index=False)

print("Saved validation matrix:", validation_matrix_path)

Saved validation matrix: /workspace/dataset/ksl_project_data/validation/healthcare_phrase_validation_matrix.csv


## 7. Create expert review questionnaire

This questionnaire is for evaluating the proposed system/framework.

It uses a 1–5 Likert scale:

| Score | Meaning |
|---|---|
| 1 | Strongly disagree |
| 2 | Disagree |
| 3 | Neutral |
| 4 | Agree |
| 5 | Strongly agree |

Reviewer groups can include:

- KSL interpreter
- deaf/KSL user
- healthcare worker
- ICT/AI expert
- academic/research expert

In [7]:
questionnaire_items = [
    {
        "section": "Relevance",
        "item_id": "REL_01",
        "question": "The proposed KSL translation framework addresses an important healthcare communication problem.",
        "reviewer_group": "all",
    },
    {
        "section": "Relevance",
        "item_id": "REL_02",
        "question": "The selected healthcare phrases are relevant to real patient-healthcare worker interactions.",
        "reviewer_group": "all",
    },
    {
        "section": "KSL Localization",
        "item_id": "KSL_01",
        "question": "The gloss/sign representation preserves the intended meaning of the healthcare phrase.",
        "reviewer_group": "ksl_interpreter,deaf_user",
    },
    {
        "section": "KSL Localization",
        "item_id": "KSL_02",
        "question": "The phrase can be naturally expressed in Kenyan Sign Language.",
        "reviewer_group": "ksl_interpreter,deaf_user",
    },
    {
        "section": "Medical Safety",
        "item_id": "MED_01",
        "question": "The phrase is medically clear and unlikely to cause harmful misunderstanding.",
        "reviewer_group": "healthcare_worker",
    },
    {
        "section": "Medical Safety",
        "item_id": "MED_02",
        "question": "High-risk phrases such as chest pain, breathing difficulty, and medicine allergy require high translation accuracy.",
        "reviewer_group": "healthcare_worker,ksl_interpreter",
    },
    {
        "section": "Usability",
        "item_id": "USE_01",
        "question": "The proposed system would be easy for healthcare workers to use during patient interaction.",
        "reviewer_group": "healthcare_worker,ict_ai_expert",
    },
    {
        "section": "Usability",
        "item_id": "USE_02",
        "question": "The proposed system would improve communication accessibility for deaf or hard-of-hearing patients.",
        "reviewer_group": "all",
    },
    {
        "section": "Technical Feasibility",
        "item_id": "TECH_01",
        "question": "The proposed use of video recognition, landmark extraction, and transformer-based translation is technically feasible.",
        "reviewer_group": "ict_ai_expert,academic_expert",
    },
    {
        "section": "Technical Feasibility",
        "item_id": "TECH_02",
        "question": "The proposed framework can be developed into a practical prototype with sufficient localized KSL training data.",
        "reviewer_group": "ict_ai_expert,academic_expert",
    },
    {
        "section": "Reliability",
        "item_id": "RELIA_01",
        "question": "The system should include human review or confirmation for high-risk healthcare phrases.",
        "reviewer_group": "all",
    },
    {
        "section": "Adoption",
        "item_id": "ADOPT_01",
        "question": "Healthcare facilities would benefit from adopting a validated KSL translation support system.",
        "reviewer_group": "all",
    },
]

questionnaire_df = pd.DataFrame(questionnaire_items)
questionnaire_df

,section,item_id,question,reviewer_group
0,Relevance,REL_01,The proposed KSL translation framework address...,all
1,Relevance,REL_02,The selected healthcare phrases are relevant t...,all
2,KSL Localization,KSL_01,The gloss/sign representation preserves the in...,"ksl_interpreter,deaf_user"
3,KSL Localization,KSL_02,The phrase can be naturally expressed in Kenya...,"ksl_interpreter,deaf_user"
4,Medical Safety,MED_01,The phrase is medically clear and unlikely to ...,healthcare_worker
5,Medical Safety,MED_02,"High-risk phrases such as chest pain, breathin...","healthcare_worker,ksl_interpreter"
6,Usability,USE_01,The proposed system would be easy for healthca...,"healthcare_worker,ict_ai_expert"
7,Usability,USE_02,The proposed system would improve communicatio...,all
8,Technical Feasibility,TECH_01,"The proposed use of video recognition, landmar...","ict_ai_expert,academic_expert"
9,Technical Feasibility,TECH_02,The proposed framework can be developed into a...,"ict_ai_expert,academic_expert"


## 8. Save expert review questionnaire

In [8]:
questionnaire_path = VALIDATION_DIR / "expert_review_questionnaire.csv"
questionnaire_df.to_csv(questionnaire_path, index=False)

print("Saved expert review questionnaire:", questionnaire_path)

Saved expert review questionnaire: /workspace/dataset/ksl_project_data/validation/expert_review_questionnaire.csv


## 9. Create validation scoring template

This template is where reviewer scores can be entered.

Each reviewer can score each questionnaire item from 1 to 5.

The columns include:

| Column | Meaning |
|---|---|
| `reviewer_id` | Anonymous reviewer ID |
| `reviewer_group` | Type of reviewer |
| `item_id` | Questionnaire item |
| `score_1_to_5` | Likert score |
| `comments` | Optional feedback |

In [9]:
reviewer_groups = [
    "ksl_interpreter",
    "deaf_user",
    "healthcare_worker",
    "ict_ai_expert",
    "academic_expert",
]

scoring_rows = []

for reviewer_group in reviewer_groups:
    for _, item in questionnaire_df.iterrows():
        allowed_groups = item["reviewer_group"]

        if allowed_groups == "all" or reviewer_group in allowed_groups.split(","):
            scoring_rows.append(
                {
                    "reviewer_id": "",
                    "reviewer_group": reviewer_group,
                    "section": item["section"],
                    "item_id": item["item_id"],
                    "question": item["question"],
                    "score_1_to_5": "",
                    "comments": "",
                }
            )

scoring_template_df = pd.DataFrame(scoring_rows)
scoring_template_df.head(20)

,reviewer_id,reviewer_group,section,item_id,question,score_1_to_5,comments
0,,ksl_interpreter,Relevance,REL_01,The proposed KSL translation framework address...,,
1,,ksl_interpreter,Relevance,REL_02,The selected healthcare phrases are relevant t...,,
2,,ksl_interpreter,KSL Localization,KSL_01,The gloss/sign representation preserves the in...,,
3,,ksl_interpreter,KSL Localization,KSL_02,The phrase can be naturally expressed in Kenya...,,
4,,ksl_interpreter,Medical Safety,MED_02,"High-risk phrases such as chest pain, breathin...",,
5,,ksl_interpreter,Usability,USE_02,The proposed system would improve communicatio...,,
6,,ksl_interpreter,Reliability,RELIA_01,The system should include human review or conf...,,
7,,ksl_interpreter,Adoption,ADOPT_01,Healthcare facilities would benefit from adopt...,,
8,,deaf_user,Relevance,REL_01,The proposed KSL translation framework address...,,
9,,deaf_user,Relevance,REL_02,The selected healthcare phrases are relevant t...,,


## 10. Save validation scoring template

In [10]:
scoring_template_path = VALIDATION_DIR / "validation_scoring_template.csv"
scoring_template_df.to_csv(scoring_template_path, index=False)

print("Saved scoring template:", scoring_template_path)

Saved scoring template: /workspace/dataset/ksl_project_data/validation/validation_scoring_template.csv


## 11. Create example reviewer responses

This cell creates synthetic example data to demonstrate how analysis will work.

Important:

```text
These are dummy/example responses only.
Do not use them as real validation findings.
```

Once you collect real responses, replace this dummy data with actual scores.

In [11]:
np.random.seed(42)

example_rows = []

example_reviewer_ids = {
    "ksl_interpreter": ["KI_01", "KI_02"],
    "deaf_user": ["DU_01", "DU_02"],
    "healthcare_worker": ["HW_01", "HW_02"],
    "ict_ai_expert": ["ICT_01", "ICT_02"],
    "academic_expert": ["AC_01"],
}

for reviewer_group, reviewer_ids in example_reviewer_ids.items():
    for reviewer_id in reviewer_ids:
        for _, item in questionnaire_df.iterrows():
            allowed_groups = item["reviewer_group"]

            if allowed_groups == "all" or reviewer_group in allowed_groups.split(","):
                score = np.random.choice([3, 4, 5], p=[0.15, 0.45, 0.40])

                example_rows.append(
                    {
                        "reviewer_id": reviewer_id,
                        "reviewer_group": reviewer_group,
                        "section": item["section"],
                        "item_id": item["item_id"],
                        "question": item["question"],
                        "score_1_to_5": int(score),
                        "comments": "",
                    }
                )

example_scores_df = pd.DataFrame(example_rows)
example_scores_df.head()

,reviewer_id,reviewer_group,section,item_id,question,score_1_to_5,comments
0,KI_01,ksl_interpreter,Relevance,REL_01,The proposed KSL translation framework address...,4,
1,KI_01,ksl_interpreter,Relevance,REL_02,The selected healthcare phrases are relevant t...,5,
2,KI_01,ksl_interpreter,KSL Localization,KSL_01,The gloss/sign representation preserves the in...,5,
3,KI_01,ksl_interpreter,KSL Localization,KSL_02,The phrase can be naturally expressed in Kenya...,4,
4,KI_01,ksl_interpreter,Medical Safety,MED_02,"High-risk phrases such as chest pain, breathin...",4,


## 12. Analyze validation scores

This calculates:

| Metric | Meaning |
|---|---|
| Mean score | Average agreement |
| Standard deviation | Variation in reviewer opinions |
| Percentage agreement | Percentage scoring 4 or 5 |
| Number of responses | Sample size per item/section |

Again, the current data is example/dummy data only.

In [12]:
def summarize_scores(scores_df):
    df = scores_df.copy()
    df["score_1_to_5"] = pd.to_numeric(df["score_1_to_5"], errors="coerce")

    item_summary = (
        df.groupby(["section", "item_id", "question"])
        .agg(
            mean_score=("score_1_to_5", "mean"),
            std_score=("score_1_to_5", "std"),
            responses=("score_1_to_5", "count"),
            agreement_count=("score_1_to_5", lambda x: (x >= 4).sum()),
        )
        .reset_index()
    )

    item_summary["percentage_agreement"] = (
        item_summary["agreement_count"] / item_summary["responses"] * 100
    )

    section_summary = (
        df.groupby("section")
        .agg(
            mean_score=("score_1_to_5", "mean"),
            std_score=("score_1_to_5", "std"),
            responses=("score_1_to_5", "count"),
            agreement_count=("score_1_to_5", lambda x: (x >= 4).sum()),
        )
        .reset_index()
    )

    section_summary["percentage_agreement"] = (
        section_summary["agreement_count"] / section_summary["responses"] * 100
    )

    return item_summary, section_summary


item_summary_df, section_summary_df = summarize_scores(example_scores_df)

print("Item-level summary")
display(item_summary_df)

print("Section-level summary")
display(section_summary_df)

Item-level summary


,section,item_id,question,mean_score,std_score,responses,agreement_count,percentage_agreement
0,Adoption,ADOPT_01,Healthcare facilities would benefit from adopt...,4.000000,0.866025,9,6,66.666667
1,KSL Localization,KSL_01,The gloss/sign representation preserves the in...,4.250000,0.957427,4,3,75.000000
2,KSL Localization,KSL_02,The phrase can be naturally expressed in Kenya...,4.250000,0.500000,4,4,100.000000
3,Medical Safety,MED_01,The phrase is medically clear and unlikely to ...,3.000000,0.000000,2,0,0.000000
4,Medical Safety,MED_02,"High-risk phrases such as chest pain, breathin...",4.500000,0.577350,4,4,100.000000
5,Relevance,REL_01,The proposed KSL translation framework address...,4.444444,0.527046,9,9,100.000000
6,Relevance,REL_02,The selected healthcare phrases are relevant t...,4.333333,0.500000,9,9,100.000000
7,Reliability,RELIA_01,The system should include human review or conf...,4.000000,0.707107,9,7,77.777778
8,Technical Feasibility,TECH_01,"The proposed use of video recognition, landmar...",4.000000,1.000000,3,2,66.666667
9,Technical Feasibility,TECH_02,The proposed framework can be developed into a...,4.000000,1.000000,3,2,66.666667


Section-level summary


,section,mean_score,std_score,responses,agreement_count,percentage_agreement
0,Adoption,4.000000,0.866025,9,6,66.666667
1,KSL Localization,4.250000,0.707107,8,7,87.500000
2,Medical Safety,4.000000,0.894427,6,4,66.666667
3,Relevance,4.388889,0.501631,18,18,100.000000
4,Reliability,4.000000,0.707107,9,7,77.777778
5,Technical Feasibility,4.000000,0.894427,6,4,66.666667
6,Usability,4.153846,0.688737,13,11,84.615385


## 13. Save example analysis outputs

These files show the expected analysis format.

When real reviewer data is available, replace the example data and rerun the analysis.

In [13]:
example_scores_path = VALIDATION_DIR / "example_validation_scores_dummy.csv"
item_summary_path = VALIDATION_DIR / "example_item_summary_dummy.csv"
section_summary_path = VALIDATION_DIR / "example_section_summary_dummy.csv"

example_scores_df.to_csv(example_scores_path, index=False)
item_summary_df.to_csv(item_summary_path, index=False)
section_summary_df.to_csv(section_summary_path, index=False)

print("Saved example scores:", example_scores_path)
print("Saved item summary:", item_summary_path)
print("Saved section summary:", section_summary_path)

Saved example scores: /workspace/dataset/ksl_project_data/validation/example_validation_scores_dummy.csv
Saved item summary: /workspace/dataset/ksl_project_data/validation/example_item_summary_dummy.csv
Saved section summary: /workspace/dataset/ksl_project_data/validation/example_section_summary_dummy.csv


## 14. Create final validation summary template

This is a blank summary template for the thesis/report.

You can fill it after collecting real reviewer responses.

In [14]:
summary_rows = []

for section in questionnaire_df["section"].unique():
    summary_rows.append(
        {
            "validation_dimension": section,
            "mean_score": "",
            "percentage_agreement": "",
            "main_positive_feedback": "",
            "main_concerns": "",
            "recommendation": "",
            "status": "pending",
        }
    )

validation_summary_template_df = pd.DataFrame(summary_rows)
validation_summary_template_df

,validation_dimension,mean_score,percentage_agreement,main_positive_feedback,main_concerns,recommendation,status
0,Relevance,,,,,,pending
1,KSL Localization,,,,,,pending
2,Medical Safety,,,,,,pending
3,Usability,,,,,,pending
4,Technical Feasibility,,,,,,pending
5,Reliability,,,,,,pending
6,Adoption,,,,,,pending


## 15. Save validation summary template

In [15]:
summary_template_path = VALIDATION_DIR / "validation_summary_template.csv"
validation_summary_template_df.to_csv(summary_template_path, index=False)

print("Saved validation summary template:", summary_template_path)

Saved validation summary template: /workspace/dataset/ksl_project_data/validation/validation_summary_template.csv


## 16. Create validation status rules

These rules help decide whether a phrase or system dimension is accepted.

Suggested interpretation:

| Mean score | Agreement | Status |
|---:|---:|---|
| ≥ 4.0 | ≥ 75% | Accepted |
| 3.5–3.9 | ≥ 60% | Accepted with revision |
| < 3.5 | < 60% | Needs major revision |

For high-risk clinical phrases, acceptance should be stricter.

In [16]:
validation_rules = {
    "general_acceptance": {
        "accepted": {
            "mean_score_minimum": 4.0,
            "percentage_agreement_minimum": 75,
        },
        "accepted_with_revision": {
            "mean_score_minimum": 3.5,
            "percentage_agreement_minimum": 60,
        },
        "needs_major_revision": {
            "mean_score_below": 3.5,
            "percentage_agreement_below": 60,
        },
    },
    "high_risk_phrase_acceptance": {
        "required_reviews": [
            "ksl_interpreter",
            "healthcare_worker",
            "deaf_user",
        ],
        "minimum_mean_score": 4.2,
        "minimum_percentage_agreement": 80,
        "requires_manual_confirmation": True,
    },
}

validation_rules_path = VALIDATION_DIR / "validation_status_rules.json"
validation_rules_path.write_text(json.dumps(validation_rules, indent=2), encoding="utf-8")

print("Saved validation rules:", validation_rules_path)
validation_rules

Saved validation rules: /workspace/dataset/ksl_project_data/validation/validation_status_rules.json


{'general_acceptance': {'accepted': {'mean_score_minimum': 4.0,
   'percentage_agreement_minimum': 75},
  'accepted_with_revision': {'mean_score_minimum': 3.5,
   'percentage_agreement_minimum': 60},
  'needs_major_revision': {'mean_score_below': 3.5,
   'percentage_agreement_below': 60}},
 'high_risk_phrase_acceptance': {'required_reviews': ['ksl_interpreter',
   'healthcare_worker',
   'deaf_user'],
  'minimum_mean_score': 4.2,
  'minimum_percentage_agreement': 80,
  'requires_manual_confirmation': True}}

## 17. Generate metadata

This metadata records what the notebook generated and how it fits into the project.

In [17]:
metadata = {
    "notebook": "07_healthcare_phrase_validation_matrix.ipynb",
    "stage": "Healthcare phrase and framework validation preparation",
    "purpose": "Prepare structured validation matrices, questionnaire items, scoring templates, and summary templates for KSL healthcare phrase validation.",
    "inputs": {
        "phrase_dataset": str(phrase_dataset_path) if phrase_dataset_path.exists() else "fallback phrase dataset created in notebook"
    },
    "outputs": {
        "validation_matrix": str(validation_matrix_path),
        "expert_review_questionnaire": str(questionnaire_path),
        "validation_scoring_template": str(scoring_template_path),
        "validation_summary_template": str(summary_template_path),
        "validation_rules": str(validation_rules_path),
        "example_scores_dummy": str(example_scores_path),
        "example_item_summary_dummy": str(item_summary_path),
        "example_section_summary_dummy": str(section_summary_path),
    },
    "reviewer_groups": reviewer_groups,
    "created_at": datetime.now().isoformat(),
    "important_note": "Dummy scores are for demonstration only and must not be reported as real validation evidence.",
}

metadata_path = VALIDATION_DIR / "validation_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved metadata:", metadata_path)
metadata

Saved metadata: /workspace/dataset/ksl_project_data/validation/validation_metadata.json


{'notebook': '07_healthcare_phrase_validation_matrix.ipynb',
 'stage': 'Healthcare phrase and framework validation preparation',
 'purpose': 'Prepare structured validation matrices, questionnaire items, scoring templates, and summary templates for KSL healthcare phrase validation.',
 'inputs': {'phrase_dataset': '/workspace/dataset/ksl_project_data/language_translation/healthcare_gloss_phrase_dataset.csv'},
 'outputs': {'validation_matrix': '/workspace/dataset/ksl_project_data/validation/healthcare_phrase_validation_matrix.csv',
  'expert_review_questionnaire': '/workspace/dataset/ksl_project_data/validation/expert_review_questionnaire.csv',
  'validation_scoring_template': '/workspace/dataset/ksl_project_data/validation/validation_scoring_template.csv',
  'validation_summary_template': '/workspace/dataset/ksl_project_data/validation/validation_summary_template.csv',
  'validation_rules': '/workspace/dataset/ksl_project_data/validation/validation_status_rules.json',
  'example_scores_d

## 18. How to write this in the thesis

You can use this paragraph:

> A healthcare phrase validation matrix was developed to support expert review of the proposed KSL healthcare translation framework. The matrix included selected healthcare communication phrases, their draft gloss representations, clinical domains, priority levels, risk categories, and review fields for KSL interpreters, deaf/KSL users, healthcare workers, and ICT/AI experts. A structured questionnaire was also prepared using Likert-scale items covering relevance, KSL localization, medical safety, usability, technical feasibility, reliability, and adoption readiness. The validation design supports descriptive analysis through mean scores, standard deviation, percentage agreement, and thematic review of open-ended comments.

## What has been achieved

This notebook prepared the validation layer:

```text
healthcare phrases
→ validation matrix
→ expert questionnaire
→ scoring template
→ summary template
```

## What comes next

The next notebook should be:

```text
08_text_to_sign_generation_design.ipynb
```

That notebook will cover the reverse direction:

```text
doctor healthcare text
→ KSL gloss
→ sign/avatar generation design
```

This is where diffusion/avatar generation enters the project architecture.